# **Fine-Tuning a Pretrained BERT model**

## **Loading Dataset**

Dataset "rotten_tomatoes"

Dataset from Hugging Face

Movies reviews

Labels : 0 (negative), 1 (positive)

train_data =  dataframe with 8530 rows, two columns 'text, 'label'

test_data = dataframe with 1066 rows, , two columns 'text, 'label'

In [ ]:
from datasets import load_dataset
data = load_dataset("rotten_tomatoes")
train_data = data["train"]
test_data = data["test"]

README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [ ]:
from pprint import pprint
print(type(data))
print(data)
print("\n"*2+"Train data")
pprint(train_data[:2])
print("\n"*2+"Test data")
pprint(test_data[:2])

<class 'datasets.dataset_dict.DatasetDict'>
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})


Train data
{'label': [1, 1],
 'text': ['the rock is destined to be the 21st century\'s new " conan " and '
          "that he's going to make a splash even greater than arnold "
          'schwarzenegger , jean-claud van damme or steven segal .',
          'the gorgeously elaborate continuation of " the lord of the rings " '
          'trilogy is so huge that a column of words cannot adequately '
          "describe co-writer/director peter jackson's expanded vision of j . "
          "r . r . tolkien's middle-earth ."]}


Test data
{'label': [1, 1],
 'text': ['lovingly photographed in the manner of a golden book sprung to life '
          ', stuart little 2 ma

In [ ]:
train_data.filter(lambda example: example['label'] == 1)

Filter:   0%|          | 0/8530 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label'],
    num_rows: 4265
})

In [ ]:
print(set(train_data['label']))

{0, 1}


# Classifier without fine-tuning

We create a pipeline with transformers.

2 inputs:

1.   The task
2.   The model




In [ ]:
from transformers import pipeline
pipe = pipeline(
    "text-classification",
    model="bert-base-cased"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
texts=test_data['text']
labels = test_data['label']


In [ ]:
predictions = pipe(list(texts),batch_size=32)

In [ ]:
from sklearn.metrics import f1_score,accuracy_score, classification_report
label2id = {"LABEL_0": 0,"LABEL_1": 1}
preds=[label2id[pred['label']] for pred in predictions]
print(f"Unique predictions: {set(preds)}")  # likely only {0} or {1}
print(f"F1 Score : {f1_score(labels,preds,average='macro'):.4f}")
print(f"accuracy score : {accuracy_score(labels,preds):.4f}")
print(classification_report(labels,preds,zero_division=0))

Unique predictions: {0}
F1 Score : 0.3333
accuracy score : 0.5000
              precision    recall  f1-score   support

           0       0.50      1.00      0.67       533
           1       0.00      0.00      0.00       533

    accuracy                           0.50      1066
   macro avg       0.25      0.50      0.33      1066
weighted avg       0.25      0.50      0.33      1066



# **Fine-tuning a LLM for text classification**


**Step 1: Defining the model and tokenizer**

We select the model

We import

- **the model**: the option num_labels=2 indicates we are dealing with 2 possible outcomes

The loaded BERT model **automatically** construct a structure with a head capable to deal with 2 classification labels

**AutoModelForSequenceClassification**


*   We call the model for classification. The classification head on top is randomly initialized
*   Generic, model-agnostic wrapper
*   Automatically infers the correct model class from the checkpoint name or config
*   works with BERT, RoBERTa, DistilBERT, XLN, ALBERT,...
*   Recommended, flexible approach


- **the tokenizer** associated to the model


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
model_name = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Note that the parameters of the classification heads are missing (randomly initialized) and need to be fine-tuned

In [ ]:
from transformers import AutoConfig
config = AutoConfig.from_pretrained('bert-base-cased')
print(config)
print(config.architectures)

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 28996
}

['BertForMaskedLM']


Alternative specialized functions for BERT model only


*   BERT-specific class from Hugging Face Transformers
*   Coded for the BERT architecture



In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
model_name = "bert-base-cased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name,num_labels=2)

**We tokenize our data**

A DataCollator is a class that helps building batches of data

The data collator is a data processing pattern frequently used in Hugging Face

In [ ]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

In [ ]:
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function , batched=True)

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [ ]:
cols = ["input_ids", "attention_mask", "token_type_ids", "label"]
tokenized_train.set_format(type="torch", columns=cols)
tokenized_test.set_format(type="torch", columns=cols)

Step 4: We define an **Evaluation function**

In [ ]:
import numpy as np
from sklearn.metrics import f1_score,accuracy_score, classification_report

def compute_metrics(eval_pred):
    logits, labels=eval_pred
    predictions = np.argmax(logits, axis=-1)
    return{
        "accuracy": accuracy_score(labels,predictions),
        "f1_macro": f1_score(labels,predictions,average="macro"),
        "f1_binary": f1_score(labels,predictions,average="binary")
    }

# **Training Process**

Hugging Face's Trainer class can be used to process to training

We define the **training parameters**

The **TrainingArguments** class is used to define the hyperparameters

In [ ]:
from transformers import TrainingArguments, Trainer

training_args=TrainingArguments(
    "model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=10,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=10,
    metric_for_best_model="f1_macro",
    load_best_model_at_end=True,
    report_to="none"
)

**Trainer** is used to execute the training program

In [ ]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

We train the model

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Binary
1,0.044998,0.878846,0.844278,0.844119,0.849091
2,0.072731,0.836793,0.850844,0.850775,0.847555
3,0.026024,1.015433,0.839587,0.839576,0.840930
4,0.022162,1.119908,0.839587,0.839525,0.836364
5,0.075317,1.188197,0.839587,0.839301,0.832517
6,0.000104,1.176180,0.854597,0.854550,0.851958
7,0.002704,1.244960,0.849906,0.849548,0.842209
8,0.053653,1.215503,0.856473,0.856427,0.853868
9,0.024928,1.256320,0.853659,0.853646,0.852273
10,0.000023,1.266770,0.855535,0.855533,0.854991


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=5340, training_loss=0.02331550265504045, metrics={'train_runtime': 1487.9157, 'train_samples_per_second': 57.329, 'train_steps_per_second': 3.589, 'total_flos': 2273050323914520.0, 'train_loss': 0.02331550265504045, 'epoch': 10.0})

We evaluate the model

In [ ]:
trainer.evaluate()

{'eval_loss': 1.2155694961547852,
 'eval_accuracy': 0.8574108818011257,
 'eval_f1_macro': 0.8573606724204106,
 'eval_f1_binary': 0.8546845124282982,
 'eval_runtime': 4.1319,
 'eval_samples_per_second': 257.995,
 'eval_steps_per_second': 8.229,
 'epoch': 10.0}

In [ ]:
from sklearn.metrics import classification_report

predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=-1)
print(classification_report(predictions.label_ids, preds))

NameError: name 'trainer' is not defined

In [ ]:
for log in trainer.state.log_history:
    if "eval_loss" in log:
        print(log)

{'eval_loss': 0.878846287727356, 'eval_accuracy': 0.8442776735459663, 'eval_f1_macro': 0.8441190979563074, 'eval_f1_binary': 0.8490909090909091, 'eval_runtime': 4.1404, 'eval_samples_per_second': 257.463, 'eval_steps_per_second': 8.212, 'epoch': 1.0, 'step': 534}
{'eval_loss': 0.8367926478385925, 'eval_accuracy': 0.850844277673546, 'eval_f1_macro': 0.8507748098962253, 'eval_f1_binary': 0.8475551294343241, 'eval_runtime': 4.3261, 'eval_samples_per_second': 246.413, 'eval_steps_per_second': 7.859, 'epoch': 2.0, 'step': 1068}
{'eval_loss': 1.015432596206665, 'eval_accuracy': 0.8395872420262664, 'eval_f1_macro': 0.8395758069129393, 'eval_f1_binary': 0.8409302325581396, 'eval_runtime': 5.1892, 'eval_samples_per_second': 205.426, 'eval_steps_per_second': 6.552, 'epoch': 3.0, 'step': 1602}
{'eval_loss': 1.1199076175689697, 'eval_accuracy': 0.8395872420262664, 'eval_f1_macro': 0.8395249644559672, 'eval_f1_binary': 0.8363636363636363, 'eval_runtime': 4.5802, 'eval_samples_per_second': 232.743, 

In [ ]:
trainer.state.log_history

# Saving the model

In [ ]:
output_dir = "./bert-topic-cls"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./bert-topic-cls/tokenizer_config.json', './bert-topic-cls/tokenizer.json')

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained(output_dir)
tokenizer = AutoTokenizer.from_pretrained(output_dir)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
from transformers import pipeline
model_path = "./bert-topic-cls"
classifier=pipeline("text-classification",model=model_path,tokenizer=model_path)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
sentence = [
    "This movie is absolutely boring, the story is a nonsense and the actors are bad",
    "The story is very good, the landscape are great and the actors are good"
]

prediction = classifier(sentence)
print(prediction)

[{'label': 'LABEL_0', 'score': 0.9999817609786987}, {'label': 'LABEL_1', 'score': 0.9999784231185913}]


# Freezing Layers

We can freeze some layers of the network



In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
model_name = "bert-base-cased"

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
for name, param in model.named_parameters():
  print (name)

bert.embeddings.word_embeddings.weight
bert.embeddings.position_embeddings.weight
bert.embeddings.token_type_embeddings.weight
bert.embeddings.LayerNorm.weight
bert.embeddings.LayerNorm.bias
bert.encoder.layer.0.attention.self.query.weight
bert.encoder.layer.0.attention.self.query.bias
bert.encoder.layer.0.attention.self.key.weight
bert.encoder.layer.0.attention.self.key.bias
bert.encoder.layer.0.attention.self.value.weight
bert.encoder.layer.0.attention.self.value.bias
bert.encoder.layer.0.attention.output.dense.weight
bert.encoder.layer.0.attention.output.dense.bias
bert.encoder.layer.0.attention.output.LayerNorm.weight
bert.encoder.layer.0.attention.output.LayerNorm.bias
bert.encoder.layer.0.intermediate.dense.weight
bert.encoder.layer.0.intermediate.dense.bias
bert.encoder.layer.0.output.dense.weight
bert.encoder.layer.0.output.dense.bias
bert.encoder.layer.0.output.LayerNorm.weight
bert.encoder.layer.0.output.LayerNorm.bias
bert.encoder.layer.1.attention.self.query.weight
bert.enc

We freeze the pretrained BERT model and train only the classifier

In [ ]:
for name, param in model.named_parameters():
  if "classifier" not in name:
    param.requires_grad = False

In [ ]:
from transformers import TrainingArguments, Trainer

training_args=TrainingArguments(
    "model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=10,
    metric_for_best_model="f1_macro",
    load_best_model_at_end=True,
    report_to="none"
)

In [ ]:
from transformers import TrainingArguments, Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Binary
1,0.668789,0.660988,0.598499,0.593346,0.639123
2,0.690620,0.655900,0.613508,0.613474,0.617100
3,0.678238,0.653588,0.621951,0.621855,0.615825
4,0.668495,0.652219,0.625704,0.625202,0.611490
5,0.665059,0.651731,0.624765,0.624764,0.625468


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2670, training_loss=0.6658054878649193, metrics={'train_runtime': 217.922, 'train_samples_per_second': 195.712, 'train_steps_per_second': 12.252, 'total_flos': 1136157731479560.0, 'train_loss': 0.6658054878649193, 'epoch': 5.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.6522194743156433,
 'eval_accuracy': 0.625703564727955,
 'eval_f1_macro': 0.6252019015980297,
 'eval_f1_binary': 0.611489776046738,
 'eval_runtime': 4.2293,
 'eval_samples_per_second': 252.049,
 'eval_steps_per_second': 8.039,
 'epoch': 5.0}

We freeze the first ten encoder blocks.

The 11th encoder block begins at index 165

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
for index,(name,param) in enumerate(model.named_parameters()):
  print(index,name)

0 bert.embeddings.word_embeddings.weight
1 bert.embeddings.position_embeddings.weight
2 bert.embeddings.token_type_embeddings.weight
3 bert.embeddings.LayerNorm.weight
4 bert.embeddings.LayerNorm.bias
5 bert.encoder.layer.0.attention.self.query.weight
6 bert.encoder.layer.0.attention.self.query.bias
7 bert.encoder.layer.0.attention.self.key.weight
8 bert.encoder.layer.0.attention.self.key.bias
9 bert.encoder.layer.0.attention.self.value.weight
10 bert.encoder.layer.0.attention.self.value.bias
11 bert.encoder.layer.0.attention.output.dense.weight
12 bert.encoder.layer.0.attention.output.dense.bias
13 bert.encoder.layer.0.attention.output.LayerNorm.weight
14 bert.encoder.layer.0.attention.output.LayerNorm.bias
15 bert.encoder.layer.0.intermediate.dense.weight
16 bert.encoder.layer.0.intermediate.dense.bias
17 bert.encoder.layer.0.output.dense.weight
18 bert.encoder.layer.0.output.dense.bias
19 bert.encoder.layer.0.output.LayerNorm.weight
20 bert.encoder.layer.0.output.LayerNorm.bias
21 b

In [ ]:
for index,(name,param) in enumerate(model.named_parameters()):
  if index<165:
    param.requires_grad=False

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Binary
1,0.396684,0.417982,0.813321,0.812957,0.821204
2,0.387589,0.392223,0.835835,0.835831,0.836601
3,0.302830,0.389684,0.846154,0.846110,0.843511
4,0.248432,0.414021,0.839587,0.839570,0.837915
5,0.313068,0.413158,0.840525,0.840480,0.837786


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

{'eval_loss': 0.38983863592147827,
 'eval_accuracy': 0.8461538461538461,
 'eval_f1_macro': 0.8461099687332752,
 'eval_f1_binary': 0.8435114503816794,
 'eval_runtime': 4.3013,
 'eval_samples_per_second': 247.832,
 'eval_steps_per_second': 7.905,
 'epoch': 5.0}